# 03. REINFORCE (Policy Gradient) Training on CartPole-v1

This notebook guides you through:
1. Loading standard hyperparameters from `config/reinforce.yaml`.
2. Training Policy Gradient with Return Normalization and Entropy Regularization.
3. Visualizing training learning curves across episodes.
4. Evaluating the best checkpoint on independent test seeds (100-199).

In [ ]:
import os
import sys
sys.path.insert(0, os.path.abspath('..'))
os.chdir('..')

import argparse
import yaml
import pandas as pd
import matplotlib.pyplot as plt
from libraries.reinforce import train, PolicyNetwork
from libraries.policies import get_policy
from libraries.evaluate import evaluate_policy

## 1. Load REINFORCE Hyperparameters

In [ ]:
with open('config/reinforce.yaml', 'r') as f:
    cfg = yaml.safe_load(f)
print('Default configuration:')
for k, v in cfg.items():
    print(f'  {k:20s}: {v}')

## 2. Train Agent (Seed 0)

In [ ]:
args = argparse.Namespace(**cfg)
args.seed = 0
args.tag = 'demo_nb'
args.total_steps = 40000
args.eval_every = 2500
args.eval_episodes = 10
train(args)

## 3. Visualize Training Progression

In [ ]:
df = pd.read_csv('private/logs/reinforce/demo_nb_seed0.csv')
df['ma_return'] = df['return'].rolling(window=20, min_periods=1).mean()

plt.figure(figsize=(9, 4.5))
plt.plot(df['env_step'], df['return'], alpha=0.3, color='forestgreen', label='Episode return')
plt.plot(df['env_step'], df['ma_return'], color='darkgreen', linewidth=2, label='Moving avg (20)')
plt.axhline(475, color='red', linestyle='--', label='Solved threshold (475)')
plt.title('Learning Curve - REINFORCE (Policy Gradient)')
plt.xlabel('Environment Steps')
plt.ylabel('Return')
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

## 4. Evaluate Best Checkpoint on Test Seeds (100-199)

In [ ]:
model_path = 'models/reinforce_demo_nb_seed0_best.pt'
policy = get_policy('reinforce', model_path)
eval_res = evaluate_policy(policy, n_episodes=100, seed_start=100)
print(f'Post-training mean return: {eval_res["mean"]:.2f} +- {eval_res["std"]:.2f}')
print(f'Fraction reaching max 500 score: {eval_res["fraction_500"] * 100:.1f}%')